# Регрессия SI с IC50/CC50 в признаках

Так как SI напрямую вычисляется из IC50 и CC50, то мы оставим IC50 и CC50 в признаках.  
Модель может легко восстановить эту формулу.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Загрузка данных
df = pd.read_excel('../data/Данные_для_курсовои_Классическое_МО.xlsx')
df = df.drop(columns=['Unnamed: 0'])
df = df.dropna()

In [3]:
# Подготовка данных (аккуратно уберём выбросы)

upper = df['SI'].quantile(0.99)
df['SI'] = df['SI'].clip(upper=upper)

X = df.drop(columns=['SI'])
y = df['SI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# Удаляем коррелированные признаки с порогом 0.95 (только на train)
corr = X_train.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(abs(upper[col]) > 0.95)]

# Применяем удаление и к train, и к test
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Удалено признаков: {len(to_drop)}")

Удалено признаков: 33


Применим следующие модели:  
- "LinearRegression"
- "RandomForest"
- "GradientBoosting"
- "CatBoost"
- "XGBoost"

**Подбор гиперпараметров не проводил из-за высокой вычислительной стоимости: полный перебор занимал более 3 часов.**

In [5]:
# ВСЕ 5 МОДЕЛЕЙ
models = {
    "LinearRegression": Pipeline([
        ('scaler', RobustScaler()),       # Логистическая регрессия чувствительна к масштабу
        ('model', LinearRegression())
    ]),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_seed=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
}

# Обучение и сравнение
results = []

for name, model in models.items():
    print(f"Обучение {name}...")
    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)
    y_true = y_test
    
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred)
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)

print("\n" + "="*48)
print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ ВСЕХ 5 МОДЕЛЕЙ:")
print("="*48)
print(results_df.to_string(index=False))

Обучение LinearRegression...
Обучение RandomForest...
Обучение GradientBoosting...
Обучение CatBoost...
Обучение XGBoost...

РЕЗУЛЬТАТЫ СРАВНЕНИЯ ВСЕХ 5 МОДЕЛЕЙ:
           model        MAE        RMSE          R2
GradientBoosting   7.325817   36.031163    0.916104
        CatBoost   9.093528   42.663104    0.882378
         XGBoost   7.920515   48.362698    0.848852
    RandomForest   9.582952   52.193988    0.823955
LinearRegression 157.696545 1464.282103 -137.558139


**Итог**  

GradientBoosting — лучшая модель  

MAE = 7.33 → минимальная ошибка  
RMSE = 36.03 → лучший результат  
R² = 0.916 → очень высокий  

Модель объясняет ~91.6% дисперсии — это отличный результат

**Вывод**

Наилучшие результаты продемонстрировала модель GradientBoosting, обеспечившая минимальные значения MAE и RMSE, а также максимальный коэффициент детерминации (R² = 0.916). Это свидетельствует о высокой способности модели описывать зависимость между признаками и целевой переменной. Модели CatBoost и XGBoost также показали высокое качество, однако уступили GradientBoosting. Модель RandomForest продемонстрировала несколько худшие результаты. Линейная регрессия оказалась неэффективной, что подтверждается отрицательным значением R², указывающим на невозможность описания данных линейной зависимостью.